# **Multiple-objective optimization - Evolutionary Optimization**

<b>Information on group members:</b><br>
1) 156035, Kuba Czech <br>
2) 156045, Wojciech Nagórka

## **1. Introduction**

- This script is for those who want to improve their final grade from 3.0 to 4.0. 
- Your task is to implement any one evolutionary algorithm for multiple-objective optimization introduced during the lecture (NSGA-II/NSGA-III/MOEA/D; except for NSGA).
- Note that it has to be your implementation (using external libraries is forbidden; EXCEPTION: you can use the JECDM framework: https://jecdm.cs.put.poznan.pl -- but it is a relatively complex software, and much effort must be spent to understand how to use it).
- The problem to be solved is the portfolio optimization tackled during lab 1.
- You can use the same data and price predictions as you made for lab 1 (Bundle1.zip) or update them accordingly to the next stage if you participate in the portfolio game (it is up to you).
- Apart from the two-objective scenario, tackle also a three-objective one. As for the third objective, think about some reasonable risk-measure. E.g., you can maximize the number of non-zero weights, which should refer to minimizing risk by diversifying investments.
- Perform experimental evaluation of your implementation. You can use, e.g., the IDG or the HV metric to quantify the quality of populations constructed by the method.
- The experimental evaluation should be "reasonably extensive." E.g., run your method multiple times and average the results, show average convergence plots, do the sensitivity analysis (just four combinations of population size/generations will be enough), and depict some final populations. Also, compare the populations (only for 2D scenarios) with those generated by the ECM or WSM algorithm. Note that ECM and WSM already generate Pareto optimal solutions, so these can be considered good benchmarks for comparison.
- You can report your results here, i.e., in the jupyter notebook. You do not need to prepare any pdf report, etc. 

Pytania do MT:
* Jak generowac initial population
* Mutacja i crossover (czy lepiej arytmetyczny czy klasyczny)
* Co z normalizacją

## **2. Reading data and plotting real prices**

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations_with_replacement

## **3. Price estimation (recap from Report 1)**

## **4. Risk estimation (recap from Report 1)**

## **5. NSGA-III implementation**

In [ ]:
class NSGAIII:
    def __init__(self, prices, risk_matrix, pop_size, nr_of_iterations, n_objectives, p, mutation_prob=0.1, crossover_prob=0.8):
        self.prices = prices
        self.risk_matrix = risk_matrix
        self.scores = []
        self.n_objectives = n_objectives

        self.population = []
        self.pop_size = pop_size
        self.nr_of_iterations = nr_of_iterations
        self.n = len(prices) # number of assets
        self.p = p
        self.alpha = 0.7

        self.mutation_prob = mutation_prob
        self.crossover_prob = crossover_prob

        self.reference_points = self.generate_reference_points()

    def get_population(self):
        return self.population

    def generate_reference_points(self):
        # Generate reference points uniformly on the unit simplex
        ref_points = []
        for combo in combinations_with_replacement(range(self.p + 1), self.n_objectives):
            if sum(combo) == self.p:
                ref_points.append([c / self.p for c in combo])
        return np.array(ref_points)

    def create_random_solution(self):
        return np.random.dirichlet([1] * self.n)

    def create_initial_population(self):
        assert self.population is None
        for _ in range(self.pop_size):
            new_sol = self.create_random_solution()
            self.population.append(new_sol)

    def mutation(self, sol):
        sol = np.copy(sol)
        idx1, idx2 = random.sample(range(self.n), 2)
        sol[idx1], sol[idx2] = sol[idx2], sol[idx1]
        return sol

    def crossover(self, sol1, sol2):
        # Arithmetic crossover: new_sol = alpha * sol1 + (1 - alpha) * sol2
        # child1 = self.alpha * np.copy(sol1) + (1 - self.alpha) * np.copy(sol2)
        # child2 = self.alpha * np.copy(sol2) + (1 - self.alpha) * np.copy(sol1)

        # One-point crossover
        idx = random.randint(1, self.n - 1)
        child1 = np.concatenate((sol1[:idx], sol2[idx:]))
        child2 = np.concatenate((sol2[:idx], sol1[idx:]))
        return child1/sum(child1), child2/sum(child2)
    
    def create_offspring(self, sol1, sol2):
        if random.random() < self.crossover_prob:
            child1, child2 = self.crossover(sol1, sol2)
        else:
            child1 = np.copy(sol1)
            child2 = np.copy(sol2)

        if random.random() < self.mutation_prob:
            child1 = self.mutation(child1)

        if random.random() < self.mutation_prob:
            child2 = self.mutation(child2)

        return child1, child2

    def fitness_function(self, sol):
        # to be overridden in subclass
        pass

    def evaluate_population(self):
        self.scores = []
        for sol in self.population:
            score = self.fitness_function(sol)
            self.scores.append(score)

    def dominates(self, sol1, sol2):
        # to be overridden in subclass
        pass

    def find_pareto_fronts(self):
        n = len(self.population)
        domination_counts = [0] * n
        dominated_solutions = [set() for _ in range(n)]
        fronts = [[]]

        # Calculate domination relations
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                if self.dominates(self.population[i], self.population[j]):
                    dominated_solutions[i].add(j)
                elif self.dominates(self.population[j], self.population[i]):
                    domination_counts[i] += 1

            if domination_counts[i] == 0:
                fronts[0].append(i)

        # Build subsequent fronts
        current_front = 0
        while fronts[current_front]:
            next_front = []
            for i in fronts[current_front]:
                for j in dominated_solutions[i]:
                    domination_counts[j] -= 1
                    if domination_counts[j] == 0:
                        next_front.append(j)
            current_front += 1
            if next_front:
                fronts.append(next_front)
            else:
                break

        return fronts

    def normalize_population(self):
        # Normalize the population's objective values to [0, 1] range
        scores_array = np.array(self.scores)
        min_scores = np.min(scores_array, axis=0)
        max_scores = np.max(scores_array, axis=0)
        normalized_scores = (scores_array - min_scores) / (max_scores - min_scores + 1e-9)
        self.scores = [tuple(i) for i in normalized_scores.tolist()]

    def choose_new_population(self, fronts):
        # Placeholder for selection logic based on fronts and reference points
        # This should implement the niching and selection mechanism of NSGA-III
        raise NotImplementedError("choose_new_population method is not implemented yet.")

    def evolve(self):
        self.create_initial_population()
        for _ in range(self.nr_of_iterations):
            new_population = []
            while len(new_population) < self.pop_size:
                parent1, parent2 = random.sample(self.population, 2)
                child1, child2 = self.create_offspring(parent1, parent2)
                new_population.append(child1)
                new_population.append(child2)

            self.population.extend(new_population)
            self.evaluate_population()

            fronts = self.find_pareto_fronts()
            self.normalize_population()

            self.population = self.choose_new_population(fronts)
        return self.population

In [ ]:
class NSGAIIITwoObjectives(NSGAIII):
    def __init__(self, prices, risk_matrix, pop_size, nr_of_iterations, mutation_prob=0.1, crossover_prob=0.8):
        super().__init__(prices, risk_matrix, pop_size, nr_of_iterations, 2, mutation_prob, crossover_prob)
    
    def dominates(self, sol1, sol2):
        eval1 = self.fitness_function(sol1)
        eval2 = self.fitness_function(sol2)
        if eval1[0] < eval2[0] and eval1[1] <= eval2[1]:
            return True
        if eval1[0] <= eval2[0] and eval1[1] < eval2[1]:
            return True
        return False

    def fitness_function(self, sol):
        risk = sol @ self.risk_matrix @ sol
        price = sol @ self.prices
        return -price, risk
    
class NSGAIIIThreeObjectives(NSGAIII):
    def __init__(self, prices, risk_matrix, pop_size, nr_of_iterations, eps=0.01, mutation_prob=0.1, crossover_prob=0.8):
        super().__init__(prices, risk_matrix, pop_size, nr_of_iterations, 3, mutation_prob, crossover_prob)
        self.eps = eps

    def dominates(self, sol1, sol2):
        eval1 = self.fitness_function(sol1)
        eval2 = self.fitness_function(sol2)
        if eval1[0] < eval2[0] and eval1[1] <= eval2[1] and eval1[2] <= eval2[2]:
            return True
        if eval1[0] <= eval2[0] and eval1[1] < eval2[1] and eval1[2] <= eval2[2]:
            return True
        if eval1[0] <= eval2[0] and eval1[1] <= eval2[1] and eval1[2] < eval2[2]:
            return True
        return False

    def fitness_function(self, sol):
        risk = sol @ self.risk_matrix @ sol
        price = sol @ self.prices
        diversity_coeff = len(set(np.where(sol > self.eps)[0])) / self.n
        return -price, risk, -diversity_coeff